In [4]:
from xmlrpc.client import Fault

import pandas as pd
from pathlib import Path

path = Path("/Users/michal/PycharmProjects/Stock Scraper/Index data/S&P 500 Historical Data from 1980.csv")

# plik jest rozdzielony średnikami, liczby mają separator tysięcy „,”
df = (pd.read_csv(path, sep=";")
        .rename(columns=str.strip)        # porządek w nazwach kolumn
     )

# Porządkowanie typów danych  ─────────────────────────────────────────────
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")   # 05/13/2025 → 2025-05-13

# Zamieniamy kolumny liczbowe na float (usuwamy przecinki tysięcy)
num_cols = ["Price", "Open", "High", "Low"]
df[num_cols] = (df[num_cols]
                .replace(",", "", regex=True)
                .astype(float))

# Chronologiczne sortowanie (przyda się do cummax)  ──────────────────────
df = df.sort_values("Date").reset_index(drop=True)

# Wyznaczenie ATH i daty ostatniego ATH  ─────────────────────────────────
df["ATH_price"] = df["High"].cummax()                         # najwyższa wartość do TEGO dnia
df["ATH_date"]  = df["Date"].where(df["High"] == df["ATH_price"]).ffill()

# Nowe kolumny: dni od ATH i % odchylenie od ATH  ────────────────────────
df["days_since_ATH"] = (df["Date"] - df["ATH_date"]).dt.days
df["pct_from_ATH"]   = (df["Price"] - df["ATH_price"]) / df["ATH_price"] * 100

In [5]:
df.tail(5)

,Date,Price,Open,High,Low,Vol.,Change %,ATH_price,ATH_date,days_since_ATH,pct_from_ATH
11545,2025-10-10,6552.51,6740.49,6762.40,6550.78,NaN,-2.71%,6764.58,2025-10-09,1,-3.135006
11546,2025-10-13,6655.10,6629.88,6670.15,6620.71,NaN,1.57%,6764.58,2025-10-09,4,-1.618430
11547,2025-10-14,6668.24,6591.96,6680.43,6552.59,NaN,0.20%,6764.58,2025-10-09,5,-1.424183
11548,2025-10-15,6664.97,6692.29,6725.83,6611.66,NaN,0.31%,6764.58,2025-10-09,6,-1.472523
11549,2025-10-16,6629.07,6689.02,6709.34,6593.99,NaN,-0.63%,6764.58,2025-10-09,7,-2.003229


In [6]:
import numpy as np

# 1. Nominalna różnica względem ATH (USD)
df["usd_from_ATH"] = df["Price"] - df["ATH_price"]

# 2. Dzienne stopy zwrotu (%)
df["daily_return_pct"] = df["Price"].pct_change() * 100

# 3. Annualizowana zmienność (rolling 30 dni)
df["vol_30d_ann"] = (
    df["daily_return_pct"]
      .rolling(window=30, min_periods=20)      # min_periods – żeby uniknąć NaN przy starcie
      .std()
      * np.sqrt(252)                           # annualizacja
)

# 4. Zwrot 252-dniowy (≈ rok handlowy)
df["return_1y_pct"] = df["Price"].pct_change(252) * 100

# 5. Średnie kroczące i 6. dystans od średnich
for w in (50, 100, 200):
    sma_col = f"SMA_{w}"
    ema_col = f"EMA_{w}"
    dist_col = f"dist_from_SMA_{w}_pct"

    df[sma_col]  = df["Price"].rolling(w, min_periods=w//2).mean()
    df[ema_col]  = df["Price"].ewm(span=w, adjust=False).mean()
    df[dist_col] = (df["Price"] - df[sma_col]) / df[sma_col] * 100

In [7]:
# Parametry
RISK_FREE_ANNUAL = 0.0453         # 5 % rocznie – podmień na bieżącą rentowność T-Bill - dane na 10.05.2025 to 
                                  # The risk-free annual actual in the US is commonly proxied by the yield on U.S. Treasury securities, 
                                  # particularly the 10-year Treasury note. As of May 16, 2025, the 10-year Treasury yield is 4.53%. 

RF_DAILY = (1 + RISK_FREE_ANNUAL) ** (1/252) - 1

# 8.1 Sharpe & Sortino (rolling 252 dni)
excess = df["daily_return_pct"]/100 - RF_DAILY          # nadwyżka nad RF w *ułamkach* (nie w %!)
downside = np.minimum(excess, 0)

roll = 252
df["roll_sharpe"] = (
    excess.rolling(roll).mean() /
    excess.rolling(roll).std()
    * np.sqrt(252)
)

df["roll_sortino"] = (
    excess.rolling(roll).mean() /
    downside.rolling(roll).std()
    * np.sqrt(252)
)

# 8.2 Max Drawdown i Calmar (CAGR ÷ Drawdown) ───────────────────────────────
cum_max = df["Price"].cummax()
drawdown = df["Price"] / cum_max - 1            # 0, -0.01, -0.15, …
df["drawdown_pct"] = drawdown * 100

max_dd = drawdown.min()                         # najgłębszy DD w serii
years = (df["Date"].iloc[-1] - df["Date"].iloc[0]).days / 365.25
cagr = (df["Price"].iloc[-1] / df["Price"].iloc[0]) ** (1/years) - 1
calmar = cagr / abs(max_dd)

print(f"Max drawdown: {max_dd:.2%}\nCAGR: {cagr:.2%}\nCalmar ratio: {calmar:.2f}")

# 8.3 Ulcer Index (rolling 252 dni) ─────────────────────────────────────────
df["ulcer_252d"] = (
    (drawdown.rolling(252).apply(lambda x: np.sqrt((x**2).mean())))
)

Max drawdown: -56.78%
CAGR: 9.41%
Calmar ratio: 0.17


In [8]:
df.tail(5)

,Date,Price,Open,High,Low,Vol.,Change %,ATH_price,ATH_date,days_since_ATH,...,SMA_100,EMA_100,dist_from_SMA_100_pct,SMA_200,EMA_200,dist_from_SMA_200_pct,roll_sharpe,roll_sortino,drawdown_pct,ulcer_252d
11545,2025-10-10,6552.51,6740.49,6762.40,6550.78,NaN,-2.71%,6764.58,2025-10-09,1,...,6326.7671,6356.287573,3.568061,6049.50880,6123.912035,8.314744,0.551303,0.885028,-2.979247,0.048297
11546,2025-10-13,6655.10,6629.88,6670.15,6620.71,NaN,1.57%,6764.58,2025-10-09,4,...,6333.9135,6362.204651,5.070901,6052.91395,6129.197487,9.948697,0.595186,0.957971,-1.460232,0.048306
11547,2025-10-14,6668.24,6591.96,6680.43,6552.59,NaN,0.20%,6764.58,2025-10-09,5,...,6342.1498,6368.264757,5.141635,6056.05495,6134.561095,10.108644,0.616689,0.992132,-1.265673,0.048313
11548,2025-10-15,6664.97,6692.29,6725.83,6611.66,NaN,0.31%,6764.58,2025-10-09,6,...,6350.3794,6374.140108,4.953887,6059.19185,6139.838795,9.997672,0.582213,0.936434,-1.314091,0.048320
11549,2025-10-16,6629.07,6689.02,6709.34,6593.99,NaN,-0.63%,6764.58,2025-10-09,7,...,6358.6419,6379.188225,4.252922,6062.48300,6144.706767,9.345791,0.512829,0.824872,-1.845650,0.048334


In [9]:
# Zapis do pliku CSV ──────────────────────────────────────────────────────
output_path = "/Users/michal/PycharmProjects/Stock Scraper/S&P 500 Historical Data from 1980 - with metrics.csv"
df.to_csv(output_path, index=False)
print(f"Zapisano plik z metrykami do:\n{output_path}")

Zapisano plik z metrykami do:
/Users/michal/PycharmProjects/Stock Scraper/S&P 500 Historical Data from 1980 - with metrics.csv


In [15]:
import os
import time
from datetime import datetime
from typing import Dict, List

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

###############################################################################
# Konfiguracja stałych i ścieżek                                              #
###############################################################################

OUTPUT_CSV_PATH = (
    "/Users/michal/PycharmProjects/Stock Scraper/"
    "S&P 500 Historical Data from 1980.csv"
)

DRIVER_PATH = (
    "/Users/michal/.wdm/drivers/chromedriver/mac64/135.0.7049.42/"
    "chromedriver-mac-x64/chromedriver"
)

SP500_HISTORICAL_URL = "https://www.investing.com/indices/us-spx-500-historical-data"

###############################################################################
# Inicjalizacja przeglądarki                                                  #
###############################################################################

def start_driver(driver_path: str, headless: bool = True) -> webdriver.Chrome:
    """Uruchamia instancję Chrome w trybie headless (domyślnie)."""
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    service = Service(driver_path)
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_window_size(1920, 1080)
    return driver

###############################################################################
# Funkcje pomocnicze                                                          #
###############################################################################

# def _accept_cookies_if_needed(driver: webdriver.Chrome, timeout: int = 5) -> None:
#     """Kliknięcie przycisku zgody na cookies (angielski lub polski), jeśli występuje."""
#     xpath_variants = [
#         "//button[./span[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'I Accept')]]",
#         "//button[./span[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZĘĄŚŁŻŹĆŃÓ', \
#                                            'abcdefghijklmnopqrstuvwxyzęąśłżźćńó'), 'zaakceptuj')]]",
#     ]
#     for xp in xpath_variants:
#         try:
#             WebDriverWait(driver, timeout).until(
#                 EC.element_to_be_clickable((By.XPATH, xp))
#             ).click()
#             break
#         except Exception:
#             continue  # spróbuj następnego wariantu


def _reformat_date(date_str_en: str) -> str:
    """Konwertuje datę z formatu "May 21, 2025" lub "May 21,2025" na "21/05/2025"."""
    date_str_en = date_str_en.strip()
    for fmt in ("%b %d, %Y", "%B %d, %Y"):
        try:
            return datetime.strptime(date_str_en, fmt).strftime("%d/%m/%Y")
        except ValueError:
            continue
    raise ValueError(f"Nieobsługiwany format daty: {date_str_en}")


def _wait_for_first_row(driver: webdriver.Chrome, timeout: int = 5):
    """Czeka, aż pojawi się pierwszy wiersz z danymi (z tagiem <time>)."""
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr time"))
    )

###############################################################################
# Główna logika scrapera                                                      #
###############################################################################

def _extract_cells_text(cells: List[webdriver.remote.webelement.WebElement]) -> List[str]:
    return [c.text.strip() for c in cells]


def scrape_latest_row(driver: webdriver.Chrome) -> Dict[str, str]:
    """Pobiera najnowszy wiersz z tabeli danych S&P 500."""
    driver.get(SP500_HISTORICAL_URL)

    # _accept_cookies_if_needed(driver)
    _wait_for_first_row(driver)

    first_time = driver.find_element(By.CSS_SELECTOR, "table tbody tr time")
    date_en = first_time.get_attribute("datetime") or first_time.text

    first_row = first_time.find_element(By.XPATH, "ancestor::tr")
    cells = first_row.find_elements(By.TAG_NAME, "td")
    if len(cells) < 7:
        time.sleep(1)
        cells = first_row.find_elements(By.TAG_NAME, "td")

    raw_values = _extract_cells_text(cells)
    raw_values[0] = date_en

    row = {
        "Date": _reformat_date(raw_values[0]),
        "Price": raw_values[1],
        "Open": raw_values[2],
        "High": raw_values[3],
        "Low": raw_values[4],
        "Vol.": raw_values[5],
        "Change %": raw_values[6],
    }
    return row

###############################################################################
# Zapisywanie danych                                                          #
###############################################################################

def save_row_to_csv_top(row: Dict[str, str], csv_path: str) -> None:
    """Dodaje wiersz na początek pliku CSV (najnowsze dane na górze).

    Jeśli plik nie istnieje, zostanie utworzony wraz z nagłówkiem.
    """
    new_df = pd.DataFrame([row])

    if os.path.isfile(csv_path):
        existing_df = pd.read_csv(csv_path, sep=";")
        # Łączymy: najnowszy wiersz + cała reszta
        combined = pd.concat([new_df, existing_df], ignore_index=True)
    else:
        combined = new_df

    combined.to_csv(csv_path, sep=";", index=False)

###############################################################################
# Uruchomienie                                                                #
###############################################################################

def main():
    driver = start_driver(DRIVER_PATH, headless=False)
    try:
        latest_row = scrape_latest_row(driver)
        save_row_to_csv_top(latest_row, OUTPUT_CSV_PATH)
        print("Dodano wiersz (na górze):", latest_row)
    finally:
        driver.quit()


if __name__ == "__main__":
    main()



Dodano wiersz (na górze): {'Date': '21/05/2025', 'Price': '5,844.61', 'Open': '5,910.18', 'High': '5,938.37', 'Low': '5,830.91', 'Vol.': '', 'Change %': '-1.61%'}


In [34]:
import os
import time
from datetime import datetime
from typing import Dict, List

import numpy as np
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

###############################################################################
# Konfiguracja stałych i ścieżek                                              #
###############################################################################

BASE_DIR = "/Users/michal/PycharmProjects/Stock Scraper"
RAW_CSV_PATH = os.path.join(BASE_DIR, "S&P 500 Historical Data from 1980.csv")
METRICS_CSV_PATH = os.path.join(
    BASE_DIR, "S&P 500 Historical Data from 1980 - with metrics.csv"
)

DRIVER_PATH = (
    "/Users/michal/.wdm/drivers/chromedriver/mac64/135.0.7049.42/"
    "chromedriver-mac-x64/chromedriver"
)

SP500_HISTORICAL_URL = "https://www.investing.com/indices/us-spx-500-historical-data"

###############################################################################
# Inicjalizacja przeglądarki                                                  #
###############################################################################

def start_driver(driver_path: str, headless: bool = True) -> webdriver.Chrome:
    """Uruchamia sterownik Chrome (domyślnie w trybie headless)."""
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    service = Service(driver_path)
    driver = webdriver.Chrome(service=service, options=options)
    driver.set_window_size(1920, 1080)
    return driver

###############################################################################
# Funkcje pomocnicze                                                          #
###############################################################################

def _accept_cookies_if_needed(driver: webdriver.Chrome, timeout: int = 5) -> None:
    xpath_variants = [
        "//button[./span[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'accept')]]",
        "//button[./span[contains(translate(., 'ABCDEFGHIJKLMNOPQRSTUVWXYZĘĄŚŁŻŹĆŃÓ', \
                                           'abcdefghijklmnopqrstuvwxyzęąśłżźćńó'), 'zaakceptuj')]]",
    ]
    for xp in xpath_variants:
        try:
            WebDriverWait(driver, timeout).until(
                EC.element_to_be_clickable((By.XPATH, xp))
            ).click()
            break
        except Exception:
            continue


def _reformat_date(date_str_en: str) -> str:
    """Konwertuje datę 'May 21, 2025' → '21/05/2025'."""
    date_str_en = date_str_en.strip()
    for fmt in ("%b %d, %Y", "%B %d, %Y"):
        try:
            return datetime.strptime(date_str_en, fmt).strftime("%m/%d/%Y")
        except ValueError:
            continue
    raise ValueError(f"Nieobsługiwany format daty: {date_str_en}")


def _wait_for_first_row(driver: webdriver.Chrome, timeout: int = 5):
    WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "table tbody tr time"))
    )

###############################################################################
# GŁÓWNY SCRAPING                                                             #
###############################################################################

def _extract_cells_text(cells: List[webdriver.remote.webelement.WebElement]) -> List[str]:
    return [c.text.strip() for c in cells]


def scrape_latest_row(driver: webdriver.Chrome) -> Dict[str, str]:
    driver.get(SP500_HISTORICAL_URL)
    _accept_cookies_if_needed(driver)
    _wait_for_first_row(driver)

    first_time = driver.find_element(By.CSS_SELECTOR, "table tbody tr time")
    date_en = first_time.get_attribute("datetime") or first_time.text

    first_row = first_time.find_element(By.XPATH, "ancestor::tr")
    cells = first_row.find_elements(By.TAG_NAME, "td")
    if len(cells) < 7:
        time.sleep(1)
        cells = first_row.find_elements(By.TAG_NAME, "td")

    raw_values = _extract_cells_text(cells)
    raw_values[0] = date_en

    return {
        "Date": _reformat_date(raw_values[0]),
        "Price": raw_values[1],
        "Open": raw_values[2],
        "High": raw_values[3],
        "Low": raw_values[4],
        "Vol.": raw_values[5],
        "Change %": raw_values[6],
    }

###############################################################################
# ZAPIS SUROWEGO CSV                                                          #
###############################################################################

def save_row_to_csv_top(row: Dict[str, str], csv_path: str) -> None:
    """Wstawia nowy wiersz na początek pliku, eliminując ewentualne duplikaty."""
    new_df = pd.DataFrame([row])

    if os.path.isfile(csv_path):
        existing_df = pd.read_csv(csv_path, sep=";")
        combined = pd.concat([new_df, existing_df], ignore_index=True)
        combined = combined.drop_duplicates()
    else:
        combined = new_df

    combined.to_csv(csv_path, sep=";", index=False)

###############################################################################
# OBRÓBKA I METRYKI                                                           #
###############################################################################

def _parse_dates(series: pd.Series) -> pd.Series:
    # wszystkie daty są teraz w MM/DD/YYYY
    parsed = pd.to_datetime(series, format="%m/%d/%Y", errors="coerce")
    if parsed.isna().any():
        bad = series[parsed.isna()].unique()
        raise ValueError(f"Nie udało się sparsować dat: {', '.join(map(str, bad))}")
    return parsed


def compute_metrics(raw_csv: str, metrics_csv: str) -> None:
    """Wczytuje surowy CSV, liczy metryki i zapisuje rozszerzony plik (data w DD/MM/YYYY)."""
    df = pd.read_csv(raw_csv, sep=";").rename(columns=str.strip)

    # ───── Porządkowanie typów ────────────────────────────────────────────
    df["Date"] = _parse_dates(df["Date"])

    num_cols = ["Price", "Open", "High", "Low"]
    df[num_cols] = df[num_cols].replace({",": ""}, regex=True).astype(float)

    # Usuwamy duplikaty, sortujemy chronologicznie (najstarsze na górze)
    df = df.drop_duplicates().sort_values("Date").reset_index(drop=True)

    # ───── ATH & związane ────────────────────────────────────────────────
    df["ATH_price"] = df["High"].cummax()
    df["ATH_date"] = df["Date"].where(df["High"] == df["ATH_price"]).ffill()
    df["days_since_ATH"] = (df["Date"] - df["ATH_date"]).dt.days
    df["pct_from_ATH"] = (df["Price"] - df["ATH_price"]) / df["ATH_price"] * 100
    df["usd_from_ATH"] = df["Price"] - df["ATH_price"]

    # ───── Dzienne stopy zwrotu & zmienność ──────────────────────────────
    df["daily_return_pct"] = df["Price"].pct_change() * 100
    df["vol_30d_ann"] = df["daily_return_pct"].rolling(30, min_periods=20).std() * np.sqrt(252)
    df["return_1y_pct"] = df["Price"].pct_change(252) * 100

    # ───── Średnie kroczące & dystans ────────────────────────────────────
    for w in (50, 100, 200):
        sma = df["Price"].rolling(w, min_periods=w // 2).mean()
        ema = df["Price"].ewm(span=w, adjust=False).mean()
        df[f"SMA_{w}"] = sma
        df[f"EMA_{w}"] = ema
        df[f"dist_from_SMA_{w}_pct"] = (df["Price"] - sma) / sma * 100

    # ───── Sharpe / Sortino (rolling 252) ────────────────────────────────
    RF_ANNUAL = 0.0453
    RF_DAILY = (1 + RF_ANNUAL) ** (1 / 252) - 1
    excess = df["daily_return_pct"] / 100 - RF_DAILY
    downside = np.minimum(excess, 0)

    roll = 252
    df["roll_sharpe"] = excess.rolling(roll).mean() / excess.rolling(roll).std() * np.sqrt(252)
    df["roll_sortino"] = excess.rolling(roll).mean() / downside.rolling(roll).std() * np.sqrt(252)

    # ───── Max drawdown / Calmar / Ulcer ─────────────────────────────────
    cum_max = df["Price"].cummax()
    drawdown = df["Price"] / cum_max - 1
    df["drawdown_pct"] = drawdown * 100

    max_dd = drawdown.min()
    years = (df["Date"].iat[-1] - df["Date"].iat[0]).days / 365.25
    cagr = (df["Price"].iat[-1] / df["Price"].iat[0]) ** (1 / years) - 1
    calmar = cagr / abs(max_dd) if max_dd else np.nan

    df["ulcer_252d"] = drawdown.rolling(252).apply(lambda x: np.sqrt((x ** 2).mean()))

    # ───── Zapis rozszerzonego CSV ───────────────────────────────────────
    df_to_save = df.copy()
    df_to_save["Date"] = df_to_save["Date"].dt.strftime("%m/%d/%Y")  # zachowujemy format DD/MM/YYYY
    df_to_save.to_csv(metrics_csv, index=False)
    

    print(
        f"Zapisano metryki → {metrics_csv} (n={len(df)} wierszy)\n"
        f"Max drawdown: {max_dd:.2%}    CAGR: {cagr:.2%}    Calmar: {calmar:.2f}"
    )

###############################################################################
# MAIN                                                                        #
###############################################################################

def main():
    """Pełny pipeline: scrape → aktualizacja surowego CSV → metryki."""
    driver = start_driver(DRIVER_PATH, headless=True)
    try:
        latest = scrape_latest_row(driver)
        save_row_to_csv_top(latest, RAW_CSV_PATH)
        print("Dodano nowy wiersz RAW:", latest)
    finally:
        driver.quit()

    compute_metrics(RAW_CSV_PATH, METRICS_CSV_PATH)


if __name__ == "__main__":
    main()


Dodano nowy wiersz RAW: {'Date': '05/22/2025', 'Price': '5,842.01', 'Open': '5,841.26', 'High': '5,878.08', 'Low': '5,825.82', 'Vol.': '', 'Change %': '-0.04%'}
Zapisano metryki → /Users/michal/PycharmProjects/Stock Scraper/S&P 500 Historical Data from 1980 - with metrics.csv (n=11449 wierszy)
Max drawdown: -56.78%    CAGR: 9.19%    Calmar: 0.16
